# Hybrid Models for Hyperspectral Image Classification
**Author:** Valerio Massimo Carioti (Student ID 1983063)

## Project Aim

The goal of the project is to implement and evaluate the CTA-net architecture, a deep learning model that combines convolutional and attention based networks in order to classify pixels of hyperspectral images when only a limited amount of samples is available.

**Reference Paper:** *CNN-Transformer and Channel-Spatial Attention based network for
hyperspectral image classification with few samples*; C. Fu, T. Zhou, T. Guo, Q. Zhu, F. Luo, B. Du; 2025

## Colab Setup

To see the images and execure the code on Colab, run the following cells:

In [ ]:
# Install repository
!git clone https://github.com/JoJohnny0/nn-project.git

In [ ]:
# Change working directory
import os
os.chdir('nn-project')

## Theoretical Background and Key Concepts

### Hyperspectral Imaging

Hyperspectral Imaging (HSI) captures hundreds of continuous spectral bands for each pixels, coverning a wider range with higher precision with respect to traditional photography (that only considers visible light and splits it in three bands: red, green and blue).
One of the main difficulties encountered when dealing with HSI data is the scarcity of labelled samples. This, combined with the much higher number of informations available for each pixel, makes the usage of large networks unfeasable as they would immediately overfit. 

### Pixel Classification With Hyperspectral Images

Due to the fact that a dataset usually consists of a single image, image-to-image pixel classification is not possible. Also, doing it with patches would result in data-leakage. For this resons, pixel classification is obtained by extracting its surrounding patch and feeding it to the network. It is then given the whole patch the same label as the central pixel, making it an image classification problem.

### Local Features and Global Context

To improve pixel classification, a network should capture local fetures while taking in account the global context. This can be achieved by combining convolutional and attention layers.

## Implementation Details

### Datasets

#### Pavia University

This dataset is widely used as a benchmark for hyperspectral image classification. It consists of a an image taken by the ROSIS sensor (Reflective Optics System Imaging Spectrometer) flying over the over Pavia. It contains 610x340 pixels with a resolution of 1.3m and 115 spectral bands in 430-860nm wavelenght range. 12 of those bands were removed due to being too noisy.

The classes contained in the dataset are divided as follows:
| Label | Class | Samples |
|-|-|-|
| 0 | Undefined | 164624 |
| 1 | Asphalt | 6631 |
| 2 | Meadows | 18649 |
| 3 | Gravel | 2099 |
| 4 | Trees | 3064 |
| 5 | Painted metal sheets | 1345 |
| 6 | Bare Soil | 5029 |
| 7 | Bitumen | 1330 |
| 8 | Self-Blocking Bricks | 3682 |
| 9 | Shadows | 947 |

![Pavia University dataset. (a) False-color map; (b) Ground-truth map.](images/PaviaUniversity.png)

#### Indian Pines

This dataset consists of an image collected by Purdue University Research Repository (PURR) in 1992 using NASA's AVIRIS sensor (airborne visible/infrared imaging spectrometer) flying over the Indian Pines test site in North West Indiana. It contains 145x145 pixels with a resolution of 20 meters and 224 spectral bands in 400–2500nm wavelength range. 24 of those bands were removed due to being too noisy.

The classes contained in the dataset are divided as follows:
| Label | Class | Samples |
|-|-|-|
| 0 | Undefined | 10776 |
| 1 | Alfalfa | 46 |
| 2 | Corn-Notill | 1428 |
| 3 | Corn-Mintill | 830 |
| 4 | Corn | 237 |
| 5  | Grass-Pasture | 483 |
| 6 | Grass-Trees | 730 |
| 7 | Grass-Pasture-Mowed | 28 |
| 8 | Hay-Windrowed | 478 |
| 9 | Oats | 20 |
| 10 | Soybean-Notill | 972 |
| 11 | Soybean-Mintill | 2455 |
| 12 | Soybean-Clean | 593 |
| 13 | Wheat | 205 |
| 14 | Woods | 1265 |
| 15 | Buildings-Grass-Trees-Drives | 386 |
| 16 | Stone-Steel-Towers | 93 |

![Indian Pines dataset. (a) False-color map; (b) Ground-truth map](images/IndianPines.png)

This dataset provides a more difficult challenge than Pavia University due to the higher number of classes and the fact that they are far more unbalanced.

### Sample Amplification

In order to increase the number of training samples, three tecniques have been applied.
- **Random Rotation:** each image is randomly rotated, using mirror padding.
- **Random Noise:** random noise is added to the image, excluded the central pixels.
- **Averaging Images:** taken two patches of the same class, average them.

These new samples are collected before the traing phase and added to training set, together with the original ones.

### Model

The implemented model is a CTA-net and it consists of two main blocks: the mixed Convolution-Transformer (CT) block and the Attention block. Their main features are described below.

![CTA-net architecture](images/cta-net.png)

#### CT Block

The mixed Convolution-Transformer block presents two parallel branches: one containing a multi-resolution CNN and one implementing a Conformer-inspired transformer. The outputs of the two branches are concatenated and a residual connection is applied.

The multi-head self attention uses a relative positional encoding that consists of bias applied directly to the attention weights. This bias is learnable and based on the relative position of the features in the input.

![Mixed Convolution-Attention block](images/ct-block.png)

![trasnformer](images/transformer.png)

#### Attention Block

The attention block is divided into a channel attention block and a spatial attention block.
The first one learns for each channel a weight between 0 and 1 and uses it to multiply the input before applying a residual connection, while the second one does the same with spatial informations.

Those spatial informations are minimum, maximum, average and standard deviation of each pixel over the channels. They then passed through a convolutional layer and a PReLU activation function before being concatenated with the original input (all this happens in the Spatial information and Concat blocks of the figure below).

![attention](images/attention%20block.png)

### Experimental Setup

In order to test how the model would perform in a setting with very few available samples, only 15 samples per class have been used (10 for training and 5 for validation).

The data preprocessing and the network have been tested on Pavia University dataset and then applied directly to Indian Pines without further hyperparameter tuning to see how well it works on a much different dataset without changes.

A list of the hyperparameters used follows:

|Hyperparameter | Value |
|-|-|
| Side of each patch | 15 |
| Added noise standard deviation | 0.1 |
| Side of the zero-noise region | 3 |
| # Hidden channels | 128 |
| # Heads | 2 |
| Dropout | 0.1 |
| Learning rate | 8e-5 |
| Batch size | 32 |

The code has been executed on Google Colab using the T4 GPU.

## Results

## Reflections

## References

- *CNN-Transformer and Channel-Spatial Attention based network for hyperspectral image classification with few samples*; C. Fu, T. Zhou, T. Guo, Q. Zhu, F. Luo, B. Du; 2025
- *Conformer: Local Features Coupling Global Representations for Visual Recognition*; Z. Peng, W. Huang, S. Gu, L. Xie, Y. Wang, J. Jiao, Q. Ye; 2021
- *Learning Hyperspectral Feature Extraction and Classification with ResNeXt Network*; D. Nyasaka, J. Wang, H. Tinega; 2020
- [Pavia Univeristy HSI Dataset](https://www.kaggle.com/datasets/syamkakarla/pavia-university-hsi)
- [Indian Pines Hyperspectral Dataset](https://www.kaggle.com/datasets/abhijeetgo/indian-pines-hyperspectral-dataset)

## Reproducibility Instructions

**Dependencies:**
- Python 3.12+
- ipykernel, kagglehub, lightning, matplotlib, nbformat, numpy, scikit-learn, scipy, torch, torchmetrics, torchvision, wandb.

To reproduce the results, run the code below. The used seeds are those from 0 to 9.

Being both the models and the datasets very lightweight, the code will also run on CPU.

**Note:** depending on the hardware used, it may be needed to change the `precision` parameter in the [training section](#training) to `'32-true'` or `'16-mixed'`. This may lead to slightly different results.

## Code

### Setup

In [ ]:
# Install dependencies
%pip install -r requirements.txt

In [ ]:
from typing import Literal

import kagglehub
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from matplotlib.axes import Axes
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from scipy.io import loadmat
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
from torch.utils.data import DataLoader
from torchmetrics.functional import accuracy, cohen_kappa
import wandb

from modules.cta_net.cta_net import CTA_Lightning
from modules.dataset import get_loaders

In [ ]:
# Configuration
dataset: Literal['PaviaUniversity', 'IndianPines'] = 'PaviaUniversity'
seed: int|None = None

In [ ]:
# Data Hyperparameters
patch_size: int = 15
train_samples_per_class: int = 10
val_samples_per_class: int = 5
sigma: float = 0.1
central_region_size: int = 3

# Model Hyperparameters
hidden_channels: int = 128
heads: int = 2

# Training Hyperparameters
dropout: float = 0.1
lr: float = 8e-5
batch_size: int = 32
epochs: int = 150

In [ ]:
# Set random seed
if seed is not None:
    pl.seed_everything(seed)

### Data Handling

In [ ]:
# Download dataset if needed
image: NDArray[np.uint16]
labels: NDArray[np.uint8]
if dataset == 'PaviaUniversity':
    dataset_path: str = kagglehub.dataset_download('syamkakarla/pavia-university-hsi')
    image = loadmat(f'{dataset_path}/PaviaU.mat')['paviaU']
    labels = loadmat(f'{dataset_path}/PaviaU_gt.mat')['paviaU_gt']
else:
    dataset_path: str = kagglehub.dataset_download('abhijeetgo/indian-pines-hyperspectral-dataset')
    image  = np.load(f'{dataset_path}/indianpinearray.npy')
    labels = np.load(f'{dataset_path}/IPgt.npy')

In [ ]:
# Get data loaders
train_loader: DataLoader[list[torch.Tensor]]
val_loader: DataLoader[list[torch.Tensor]]
test_loader: DataLoader[list[torch.Tensor]]
train_loader, val_loader, test_loader = get_loaders(image,
                                                    labels,
                                                    patch_size,
                                                    train_samples_per_class = train_samples_per_class,
                                                    val_samples_per_class = val_samples_per_class,
                                                    sigma = sigma,
                                                    central_region_size = central_region_size,
                                                    batch_size = batch_size
                                                    )

### Training

In [ ]:
# Initialize logger
wandb_logger: WandbLogger = WandbLogger(project = f"Hybrid Models for Hyperspectral Image Classification",
                                        name = f"CTA-net_{dataset}_seed={seed}",
                                        save_dir = 'out',
                                        # Parameters not logged by the trainer
                                        config = {'train_samples_per_class': train_samples_per_class,
                                                  'val_samples_per_class': val_samples_per_class,
                                                  'sigma': sigma,
                                                  'central_region_size': central_region_size,
                                                  'batch_size': batch_size
                                                  }
                                        )
wandb_logger.experiment.define_metric('*', step_metric = 'epoch')

In [ ]:
# Add best checkpoint callback
save_best: ModelCheckpoint = ModelCheckpoint(monitor = 'val_loss',
                                             dirpath = f'out/checkpoints/{dataset}',
                                             filename = f'cta-net-epoch={{epoch}}-seed={seed}',
                                             save_weights_only = True
                                             )

In [ ]:
# Initialize model and trainer
n_classes: int = int(labels.max())
model: CTA_Lightning = CTA_Lightning(in_channels = image.shape[2],
                                     hidden_channels = hidden_channels,
                                     out_channels = n_classes,
                                     heads = heads,
                                     window_size = patch_size,
                                     dropout = dropout,
                                     lr = lr
                                     )
trainer: pl.Trainer = pl.Trainer(max_epochs = epochs,
                                 precision = 'bf16-mixed',  # change if needed
                                 callbacks = save_best,
                                 logger = wandb_logger,
                                 log_every_n_steps = len(train_loader)
                                 )

In [ ]:
# Train the model
trainer.fit(model, train_loader, val_loader)

### Test

In [ ]:
# Get predictions
preds: torch.Tensor = torch.concat(trainer.predict(model, test_loader, ckpt_path = 'best'))  # type: ignore

# Get targets
targets: torch.Tensor = torch.concat([batch[1] for batch in test_loader])

In [ ]:
# General metrics
metrics: dict[str, float] = {'Overall Accuracy': accuracy(preds, targets, task = 'multiclass', average = 'micro', num_classes = n_classes).item(),
                             'Average Accuracy': accuracy(preds, targets, task = 'multiclass', average = 'macro', num_classes = n_classes).item(),
                             'Kappa': cohen_kappa(preds, targets, task = 'multiclass', num_classes = n_classes).item()
                             }

# Class-wise accuracy
conf_matrix: NDArray[np.int64] = confusion_matrix(targets, preds)
for i, class_acc in enumerate(conf_matrix.diagonal() / conf_matrix.sum(axis = 1)):
    metrics[f'Class {i + 1} Accuracy'] = class_acc.item()

In [ ]:
# Print metrics
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# Display confusion matrix
fig: Figure
ax: Axes
fig, ax = plt.subplots(figsize = (6, 6))
disp: ConfusionMatrixDisplay = ConfusionMatrixDisplay(confusion_matrix = conf_matrix)
disp.plot(ax = ax)
ax.set_title("Confusion Matrix")
fig.tight_layout()

In [ ]:
# Log to wandb
wandb_logger.experiment.log(metrics)
wandb_logger.experiment.log({'Confusion Matrix': wandb.Image(fig)})
wandb_logger.experiment.finish()
plt.close(fig)